# Part 1: MCQ Solver — DATATHON 2026
**Team:** GenCore

This notebook contains the code to answer 10 data-driven questions for Part 1 of the competition.

In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Add src to path
sys.path.append(os.path.abspath('../'))
from src.preprocessing import load_and_merge_order_data

DATA_PATH = '../data/raw/'

## 1 — Load Data

In [2]:
# df = load_and_merge_order_data(DATA_PATH)
# df.head()

In [3]:
#Q1
df = pd.read_csv(f'{DATA_PATH}/orders.csv')
# Lấy danh sách đơn hàng duy nhât
orders = df[['customer_id', 'order_id', 'order_date']].drop_duplicates().copy()
# Ta cần chuyển đổi định dang thời gian để thực hiện các phép tính số học thời gian
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
#Sắp xếp các đơn hàng của 1 khách cạnh nhau để dễ dàng tính khoảng cách ngày giữa các đơn hàng liên tiếp
orders = orders.dropna(subset=['order_date']).sort_values(['customer_id', 'order_date'])
# Tính hiệu số
orders['gap'] = orders.groupby('customer_id')['order_date'].diff().dt.days
# Tính trung vị 
res = orders['gap'].median()
print(f"Q1: {res} ngày")

Q1: 144.0 ngày


In [ ]:
#Q2
df = pd.read_csv(f'{DATA_PATH}/products.csv')
# Tính tỷ suất lợi nhuận từng dòng (có điều kiện cogs < price)
df['margin'] = (df['price'] - df['cogs']) / df['price']
#Gom nhóm tính từng dòng rồi sắp xếp tìm ra phân khúc có tỷ suất lợi nhuận cao nhất
res = df.groupby('segment')['margin'].mean().idxmax()
print(f"Q2: {res}")


Q2: Standard


In [5]:
#Q3
df1 = pd.read_csv(f'{DATA_PATH}/products.csv')
df2 = pd.read_csv(f'{DATA_PATH}/returns.csv')
# Kết hợp category từ products vào bảng returns theo product_id
df3 = df2.merge(df1[['product_id', 'category']], on='product_id', how='left')
# Lọc các đơn thuộc danh mục Streetwear
streetwear = df3[df3['category'] == 'Streetwear']
# Tìm lý do hoàn trả xuất hiện nhiều nhất
res = streetwear['return_reason'].value_counts().idxmax()
print(f"Q3: {res}")

Q3: wrong_size


In [6]:
#Q4
df = pd.read_csv(f'{DATA_PATH}/web_traffic.csv')
# Tính trung bình bounce_rate theo từng traffic_source
res = df.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Q4: {res}")

Q4: email_campaign


In [7]:
#Q5
df = pd.read_csv(f'{DATA_PATH}/order_items.csv')
# Xác định các dòng có áp dụng khuyến mãi
promo = (df['promo_id'].notna()) & (df['promo_id'] != 'NO_PROMO')
# Tính tỷ lệ phần trăm
res = (promo.sum() / len(df)) * 100
print(f"Q5: {res:.0f}%")

Q5: 39%


/var/folders/8h/d8423sb969v7wgq1ynjcq8300000gn/T/ipykernel_15918/1819998146.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{DATA_PATH}/order_items.csv')


In [8]:
#Q6
df1 = pd.read_csv(f'{DATA_PATH}/customers.csv')
# Lấy số đơn hàng duy nhất theo từng khách hàng và gắn nhóm tuổi
df2 = pd.read_csv(f'{DATA_PATH}/orders.csv')
x = df2[['order_id', 'customer_id']].drop_duplicates().merge(
    df1[['customer_id', 'age_group']], on='customer_id', how='left'
 )
q6_data = x[x['age_group'].notna()]
# Tính số đơn/khách hàng cho từng nhóm tuổi
orders = q6_data.groupby('age_group')['order_id'].nunique()
customers = df1[df1['age_group'].notna()].groupby('age_group')['customer_id'].nunique()
res = (orders / customers).idxmax()
print(f"Q6: {res}")

Q6: 55+


In [9]:
#Q7
df1 = pd.read_csv(f'{DATA_PATH}/geography.csv')
df2 = pd.read_csv(f'{DATA_PATH}/orders.csv')
df3 = pd.read_csv(f'{DATA_PATH}/order_items.csv')
# Tính doanh thu từng dòng đơn hàng
df3['line_revenue'] = (df3['quantity'] * df3['unit_price']) - df3['discount_amount'].fillna(0)
# Gắn orders và region từ geography vào dữ liệu doanh thu
Merge = df3.merge(df2[['order_id', 'zip']], on='order_id', how='left')
Merge = Merge.merge(df1[['zip', 'region']], on='zip', how='left')
# Tìm vùng có tổng doanh thu cao nhất
res = Merge.groupby('region')['line_revenue'].sum().idxmax()
print(f"Q7: {res}")

Q7: East


/var/folders/8h/d8423sb969v7wgq1ynjcq8300000gn/T/ipykernel_15918/325906428.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df3 = pd.read_csv(f'{DATA_PATH}/order_items.csv')


In [10]:
#Q8
df = pd.read_csv(f'{DATA_PATH}/orders.csv')
# Lọc các đơn bị hủy
cancelled = df[df['order_status'] == 'cancelled']
# Tìm phương thức thanh toán xuất hiện nhiều nhất trong các đơn bị hủy
res = cancelled['payment_method'].value_counts().idxmax()
print(f"Q8: {res}")

Q8: credit_card


In [11]:
#Q9
df = pd.read_csv(f'{DATA_PATH}/order_items.csv')
df1 = pd.read_csv(f'{DATA_PATH}/returns.csv')
df2 = pd.read_csv(f'{DATA_PATH}/products.csv')
# Gắn size vào dữ liệu bán và dữ liệu trả hàng theo product_id
sales = df.merge(df2[['product_id', 'size']], on='product_id', how='left')
returns = df1.merge(df2[['product_id', 'size']], on='product_id', how='left')
# Số dòng bán ra theo size
sales_size = sales['size'].value_counts()
# Số dòng trả hàng theo size
returns_size = returns['size'].value_counts()
# Tính tỷ lệ trả hàng/bán ra theo size và lấy size cao nhất
res = (returns_size / sales_size).idxmax()
print(f"Q9: {res}")

Q9: S


/var/folders/8h/d8423sb969v7wgq1ynjcq8300000gn/T/ipykernel_15918/1192607389.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{DATA_PATH}/order_items.csv')


In [12]:
#Q10
df = pd.read_csv(f'{DATA_PATH}/payments.csv')
# Tính giá trị thanh toán trung bình theo số kỳ trả góp
avg_payment_by_installments = df.groupby('installments')['payment_value'].mean()
# Lấy số kỳ trả góp có payment_value trung bình cao nhất
res = avg_payment_by_installments.idxmax()
print(f"Q10: {res}")

Q10: 6
